# 05c — 肩胛骨運動學的「生成」：四種假設
### *Scapula kinematics generation — frozen / de Groot rhythm / thoracoscapular / dyskinesis*

Markerless（含 Kinatrax）**看不到肩胛骨**（皮下滑動 + soft-tissue artifact），所以
肩胛的自由度必須被 **生成 / 估計 (generate)**，而不是量測。這是**運動學層**的問題，
且直接改變 glenohumeral (GH) 側的運動學與負荷——正是你子研究 A 的核心。本 notebook
實作四種肩胛生成假設並以動畫/round-trip 檢查：

1. **Frozen**：鎖住肩胛座標（baseline）。
2. **de Groot 2001 scapulohumeral rhythm** 回歸：由肱骨抬舉推肩胛上旋等。
3. **Seth 2019 thoracoscapular constraint**：肩胛在胸廓橢球面上滑動。
4. **人為 dyskinesis**：擾動 rhythm（如減少 upward rotation），作為子研究 A/C 的擾動。

> ⚙️ 本 notebook 最好用 **Seth 2019 thoracoscapular shoulder model**（自 SimTK 下載，
> 見 `data/README.md`）。若模型不在，仍提供**概念性 fallback**（用回歸函數示意）。
> 注意：肩胛 coordinate 的**確切名稱依模型而定**，程式會從載入的模型列出、不硬編。


In [ ]:
# --- 讓 notebook 找得到本章的 src/ 模組 (put chapter src/ on sys.path) ---
import sys, pathlib
CHAPTER = pathlib.Path.cwd()
if CHAPTER.name == "notebooks":
    CHAPTER = CHAPTER.parent          # 允許從 notebooks/ 內啟動
sys.path.insert(0, str(CHAPTER / "src"))

import numpy as np
import matplotlib.pyplot as plt

import osim_kinematics_io as kio
print("src loaded")


## 0. 載入模型並找出肩胛座標

先嘗試載入 Seth thoracoscapular 模型；找不到就退回 `arm26`（僅能示意，無肩胛 DOF）。
然後從模型**列出**名字含 scapula/clav 的座標——不要硬編名稱。


In [ ]:
import os, pathlib
SETH = None
for c in [os.environ.get("SETH_MODEL"),
          str(CHAPTER / "data" / "ThoracoscapularShoulderModel.osim"),
          str(CHAPTER / "data" / "Seth2019_ThoracoscapularShoulder.osim")]:
    if c and pathlib.Path(c).is_file():
        SETH = c; break

HAS_SETH = SETH is not None
model_path = SETH if HAS_SETH else None
if not HAS_SETH:
    try:
        import importlib; nb05a = None  # noqa
        from pathlib import Path
        # fallback: arm26 (無肩胛，僅示意 rhythm 函數與繪圖)
        for c in [os.environ.get("ARM26"), str(CHAPTER / "data" / "arm26.osim")]:
            if c and Path(c).is_file():
                model_path = c; break
    except Exception:
        pass
print("Seth thoracoscapular model found:", HAS_SETH, "| using:", model_path)


In [ ]:
scap_coords = []
if model_path:
    model = kio.load_model(model_path)
    for c in kio.list_coordinates(model):
        low = c["name"].lower()
        if any(key in low for key in ("scapula", "scap", "clav", "abduction",
                                      "elevation", "upward", "rotation", "tilt", "winging")):
            scap_coords.append(c["name"])
    print("候選肩胛/鎖骨相關座標 (依模型而定，請人工確認):")
    for c in kio.list_coordinates(model):
        if c["name"] in scap_coords:
            print(f"  {c['name']:22s} joint={c['joint']}")
else:
    print("無模型可載入；以下用概念性回歸函數示範 (見 method 2)。")


## Method 1 — Frozen scapula (baseline)

最簡單、也是最常被隱含採用的假設：把肩胛座標**鎖住**在中立值。用
`coord.setDefaultLocked(True)`（或在 state 上 `setLocked`）。`prescribe_and_report`
會在驅動前暫時解鎖被驅動的座標，因此若要「凍結」就**不要**把肩胛放進被驅動集合。


In [ ]:
if model_path and scap_coords:
    model = kio.load_model(model_path)
    for nm in scap_coords:
        try:
            model.getCoordinateSet().get(nm).setDefaultLocked(True)
        except Exception as e:
            print("lock", nm, "->", e)
    model.finalizeConnections(); model.initSystem()
    print("已鎖住肩胛座標:", scap_coords)
else:
    print("（示意）frozen = 肩胛座標保持中立、不隨肱骨變化。")


## Method 2 — de Groot 2001 scapulohumeral rhythm 回歸

de Groot & Brand (2001) 給出以**肱骨抬舉角**為輸入、預測肩胛（與鎖骨）旋轉的 3D 回歸
[@degroot2001]。臨床上常引用的近似是「肱骨每抬 3°，約 2° 來自 GH、1° 來自
scapulothoracic」（≈2:1 rhythm），且主要反映在肩胛 **upward rotation**。下面用一個
線性 rhythm 函數示意（真實應用請代入 de Groot 完整回歸係數）：


In [ ]:
def scapular_rhythm(humeral_elev_deg, ratio=1.0/3.0, resting_upward=5.0):
    # 近似：upward rotation = 靜止值 + ratio * 肱骨抬舉 (deg)。ratio~1/3 對應 2:1 rhythm。
    return resting_upward + ratio * np.asarray(humeral_elev_deg, dtype=float)

elev = np.linspace(0, 150, 200)                 # 肱骨抬舉 0..150 deg
up_normal = scapular_rhythm(elev, ratio=1/3)
print("在 elev=120deg 時，肩胛 upward rotation ~", scapular_rhythm(120, 1/3), "deg")


在 OpenSim 中把 rhythm **內建進模型**的正統作法是 `CoordinateCouplerConstraint`：
讓某個 dependent 肩胛座標成為 independent 肱骨抬舉座標的函數（`LinearFunction` 或
`SimmSpline`）。骨架如下（座標名稱請換成你模型實際的名稱）：


In [ ]:
# --- 以 CoordinateCouplerConstraint 內建 scapulohumeral rhythm 的骨架 ---
def add_scapulohumeral_rhythm(model, indep_coord, dep_coord, slope=1/3, intercept_deg=5.0):
    import opensim as osim
    ccc = osim.CoordinateCouplerConstraint()
    ccc.setName(f"rhythm_{dep_coord}")
    ind = osim.ArrayStr(); ind.append(indep_coord)
    ccc.setIndependentCoordinateNames(ind)
    ccc.setDependentCoordinateName(dep_coord)
    # dep(rad) = slope * indep(rad) + intercept(rad)
    ccc.setFunction(osim.LinearFunction(slope, np.deg2rad(intercept_deg)))
    model.addConstraint(ccc)
    return model

print("骨架已定義。實際使用：add_scapulohumeral_rhythm(model, '<肱骨抬舉座標>', '<肩胛上旋座標>')")
print("務必 finalizeConnections()+initSystem() 後，用 prescribe_and_report 驅動肱骨、觀察肩胛跟隨。")


## Method 3 — Seth 2019 thoracoscapular constraint

Seth et al. (2019) 的模型讓肩胛沿**胸廓橢球面 (thoracic ellipsoid)** 滑動，以
scapulothoracic 接觸/約束取代自由的肩胛關節，使肩胛姿勢在生理上合理 [@seth2019]。
若你載入了該模型，肩胛的可行姿勢**已由模型的 joint/constraint 定義**——你只需驅動
肱骨相關座標，肩胛會依約束求解（`prescribe_and_report` 的 2-arg `setValue` 會呼叫
`assemble` 讓約束被滿足）。


In [ ]:
if HAS_SETH and scap_coords:
    model = kio.load_model(SETH)
    # 找一個像「肱骨抬舉」的 independent 座標名稱 (依模型而定)
    all_names = [c["name"] for c in kio.list_coordinates(model)]
    elev_like = [n for n in all_names if "elv" in n.lower() or "elev" in n.lower()]
    print("疑似肱骨抬舉座標:", elev_like)
    print("驅動它並用 model.assemble/realizePosition 讓 thoracoscapular 約束求解肩胛。")
else:
    print("未載入 Seth 模型：thoracoscapular 約束需要該模型檔，見 data/README.md 下載說明。")


## Method 4 — 人為 dyskinesis（子研究 A / C 的擾動）

把正常 rhythm **擾動**——例如減少 upward rotation（SICK scapula 的典型特徵）——即得到
一個受限的肩胛假設。這是子研究 A 的自變項之一，也是 C 的預測模擬前哨。


In [ ]:
up_dyskinesis = scapular_rhythm(elev, ratio=1/6, resting_upward=2.0)  # 上旋不足
plt.figure(figsize=(6, 3.4))
plt.plot(elev, up_normal, "g-", label="normal rhythm (~2:1)")
plt.plot(elev, up_dyskinesis, "r--", label="dyskinesis (upward rot 不足)")
plt.plot(elev, np.zeros_like(elev), "k:", label="frozen (upward=const)")
plt.xlabel("humeral elevation (deg)"); plt.ylabel("scapular upward rotation (deg)")
plt.title("四種肩胛假設下的 upward rotation vs 肱骨抬舉")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()


## 為什麼這會改變 GH，卻（在 replay 下）改不動 UCL

- **GH 側敏感**：換肩胛假設 → 改變跨肩肌肉的 moment arm / length–tension → GH joint
  reaction force 與肩部肌力**劇烈**改變。
- **UCL 側近乎不變（replay 時）**：inverse dynamics 由遠端往近端遞迴，肘 net moment
  只取決於**前臂＋手的測量運動**；保護 UCL 的 flexor–pronator 起於內上髁、**不跨肩**。
  故在**重播測量運動**時，肩胛假設碰不到 UCL——這推動子研究 C 必須改用**預測 / 反事實
  模擬**（見研究構想與 `notes.md` Section C；UCL 以 Buffi 2015 partition 作 readout [@buffi2015]）。

## 小結
- 肩胛看不到 → 必須生成；四種假設是子研究 A 的自變項。
- Rhythm 可用 `CoordinateCouplerConstraint` 內建；thoracoscapular 約束由 Seth 模型提供。
- 生成假設強烈影響 GH、但在 replay ID 下幾乎不動 UCL —— 這是全案的關鍵邏輯。
